#DEMANDAS TI

### CONFIGURAÇÃO E CARREGAMENTO DE DATASET

In [64]:
# ============================================================
# Importa as bibliotecas do Python
# ============================================================

# Importar o pandas
import pandas as pd
import numpy as np

# Importar o LabelEncoder da biblioteca scikit-learn
from sklearn.preprocessing import LabelEncoder

# Importar a função de divisão de treino e teste
from sklearn.model_selection import train_test_split

# Modelos
from sklearn.ensemble import RandomForestClassifier
from xgboost          import XGBClassifier
from sklearn.metrics  import (accuracy_score, precision_score,
                               recall_score, f1_score,
                               classification_report)

from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble        import RandomForestClassifier
from xgboost                 import XGBClassifier



In [65]:
# ============================================================
# Carregar o dataset a partir do arquivo Excel
# ============================================================

# url do arquivo no GitHub
url_dataset = "https://raw.githubusercontent.com/gilbertoag2007/machine-learning-demandas-ti/main/DEMANDAS_DOWNSTREAM_TESTE.xlsx"

# Cria um dataframe com o conteúdo do dataset
df_original = pd.read_excel(url_dataset)

# Lista as 5 primeiras colunas do dataframe.

df_original.shape
df_original.head()

,ID_DEMANDA,SISTEMA,TIPO_DEMANDA,DATA_INICIO_PREVISTA,DATA_INICIO_REALIZADA,DATA_FIM_PREVISTA,DATA_FIM_REALIZADA,STATUS_FINAL
0,1,DFE,BUG_IMPEDITIVO,22/01/2026,22/01/2026,2026-01-24,2026-01-24,DENTRO DO PRAZO
1,2,DFE,BUG_IMPEDITIVO,2026-01-05 00:00:00,2026-01-07 00:00:00,2026-01-07,2026-01-07,DENTRO DO PRAZO
2,3,DFE,BUG_IMPEDITIVO,23/02/2026,23/02/2026,2026-02-25,2026-02-25,DENTRO DO PRAZO
3,4,DFE,BUG_IMPEDITIVO,17/02/2026,17/02/2026,2026-02-19,2026-02-19,DENTRO DO PRAZO
4,5,DFE,BUG_IMPEDITIVO,13/02/2026,13/02/2026,2026-02-15,2026-02-15,DENTRO DO PRAZO


##AJUSTES INICIAIS NO DATAFRAME

In [66]:
# ============================================================
# TÉCNICA: Label Encoding (Codificação de Rótulos)
# ============================================================

# Cria uma cópia do dataframe original para preservá-lo intacto
# Todas as alterações serão feitas apenas no df_ajustado
df_ajustado = df_original.copy()

# Cria uma nova coluna numérica baseada na coluna STATUS_FINAL
# map() substitui cada valor categórico pelo número correspondente
df_ajustado['STATUS_FINAL_NUM'] = df_ajustado['STATUS_FINAL'].map({
    'ATRASO'         : 1,
    'DENTRO DO PRAZO': 0
})

# Variavel Target
target = "STATUS_FINAL_NUM"



In [67]:
# ============================================================
# TÉCNICA: One-Hot Encoding
# ============================================================

# Aplicar One-Hot Encoding na coluna SISTEMA
# pd.get_dummies() cria uma coluna binária (0 ou 1) para cada sistema único
# dtype=int garante que os valores sejam inteiros ao invés de booleanos
# Aplicar nas colunas categóricas sem ordem natural
for coluna in ['SISTEMA', 'TIPO_DEMANDA']:
    dummies = pd.get_dummies(df_ajustado[coluna], prefix=coluna, dtype=int)
    df_ajustado = pd.concat([df_ajustado, dummies], axis=1)
    df_ajustado = df_ajustado.drop(columns=[coluna])


In [68]:

# ============================================================
# CONVERTER COLUNAS DE DATA PARA DATETIME
# Necessário para realizar operações matemáticas entre datas
# ============================================================
colunas_data = [
    'DATA_INICIO_PREVISTA',
    'DATA_INICIO_REALIZADA',
    'DATA_FIM_PREVISTA',
    'DATA_FIM_REALIZADA'
]

for coluna in colunas_data:
    df_ajustado[coluna] = pd.to_datetime(
        df_ajustado[coluna], dayfirst=True, errors='coerce'
    )


In [69]:
# ============================================================
# GERAR DATA_REFERENCIA
# Ponto médio entre início e fim — simula o momento de acompanhamento de cada demanda
# ============================================================

# Calcular ponto médio como referência padrão
# usando DATA_INICIO_PREVISTA e DATA_FIM_PREVISTA
df_ajustado['DATA_REFERENCIA'] = (
    df_ajustado['DATA_INICIO_PREVISTA'] +
    (df_ajustado['DATA_FIM_PREVISTA'] - df_ajustado['DATA_INICIO_PREVISTA']) / 2
)

# Ajustar referência para demandas já iniciadas
# usar ponto médio entre DATA_INICIO_REALIZADA e DATA_FIM_PREVISTA
mask_iniciadas = df_ajustado['DATA_INICIO_REALIZADA'].notna()

df_ajustado.loc[mask_iniciadas, 'DATA_REFERENCIA'] = (
    df_ajustado.loc[mask_iniciadas, 'DATA_INICIO_REALIZADA'] +
    (
        df_ajustado.loc[mask_iniciadas, 'DATA_FIM_PREVISTA'] -
        df_ajustado.loc[mask_iniciadas, 'DATA_INICIO_REALIZADA']
    ) / 2
)

In [70]:
# ============================================================
# GERAR FEATURES NUMÉRICAS
# ============================================================

# ----------------------------------------------------------
# Duração total planejada da demanda em dias
# Indica o tamanho e complexidade da demanda
# ----------------------------------------------------------
df_ajustado['DURACAO_PREVISTA_DIAS'] = (
    df_ajustado['DATA_FIM_PREVISTA'] - df_ajustado['DATA_INICIO_PREVISTA']
).dt.days

# ----------------------------------------------------------
# Quantidade de dias de atraso no início da demanda
# Valores positivos indicam atraso no início
# Preenchido com 0 quando não há DATA_INICIO_REALIZADA
# ----------------------------------------------------------
df_ajustado['ATRASO_INICIO_DIAS'] = (
    df_ajustado['DATA_INICIO_REALIZADA'] - df_ajustado['DATA_INICIO_PREVISTA']
).dt.days.fillna(0)

# ----------------------------------------------------------
# Dias restantes até o prazo final na data de referência
# Valores negativos indicam que o prazo já foi ultrapassado
# ----------------------------------------------------------
df_ajustado['DIAS_RESTANTES'] = (
    df_ajustado['DATA_FIM_PREVISTA'] - df_ajustado['DATA_REFERENCIA']
).dt.days

# ----------------------------------------------------------
# Percentual do prazo consumido até a data de referência
# Indica o quanto do tempo planejado já foi utilizado
# ----------------------------------------------------------
df_ajustado['PERC_TEMPO_DECORRIDO'] = (
    (df_ajustado['DATA_REFERENCIA'] - df_ajustado['DATA_INICIO_PREVISTA']).dt.days /
     df_ajustado['DURACAO_PREVISTA_DIAS']
) * 100

# ----------------------------------------------------------
# Dias sem iniciar após a DATA_INICIO_PREVISTA
# clip(lower=0) evita valores negativos para demandas
# que ainda não atingiram a data de início prevista
# ----------------------------------------------------------
df_ajustado['DIAS_SEM_INICIAR'] = (
    df_ajustado['DATA_REFERENCIA'] - df_ajustado['DATA_INICIO_PREVISTA']
).dt.days.clip(lower=0)


In [71]:
# ============================================================
# GERAR FLAGS BINÁRIAS (0 ou 1)
# ============================================================

# ----------------------------------------------------------
# Flag: a demanda já foi iniciada?
# 1 = sim | 0 = não
# ----------------------------------------------------------
df_ajustado['FLAG_INICIADA'] = (
    df_ajustado['DATA_INICIO_REALIZADA'].notna()
).astype(int)

# ----------------------------------------------------------
# Flag: a demanda atrasou no início?
# 1 = começou depois do previsto | 0 = não
# ----------------------------------------------------------
df_ajustado['FLAG_ATRASO_INICIO'] = (
    df_ajustado['ATRASO_INICIO_DIAS'] > 0
).astype(int)

# ----------------------------------------------------------
# Flag: a demanda deveria ter iniciado mas ainda não iniciou?
# 1 = passou da DATA_INICIO_PREVISTA sem início registrado
# 0 = ainda dentro do prazo de início ou já iniciada
# ----------------------------------------------------------
df_ajustado['FLAG_NAO_INICIADA_NO_PRAZO'] = (
    (df_ajustado['DATA_REFERENCIA'] >= df_ajustado['DATA_INICIO_PREVISTA']) &
    (df_ajustado['DATA_INICIO_REALIZADA'].isna())
).astype(int)

In [72]:
# ============================================================
# REMOVER COLUNAS QUE NÃO DEVEM ENTRAR NO MODELO ANTES DO TREINAMENTO
# ============================================================

colunas_remover = [
    'ID_DEMANDA',
    'STATUS_FINAL',
    'DATA_INICIO_PREVISTA',
    'DATA_INICIO_REALIZADA',
    'DATA_FIM_PREVISTA',
    'DATA_FIM_REALIZADA',
    'DATA_REFERENCIA'
]
df_ajustado = df_ajustado.drop(columns=colunas_remover)

In [73]:
# Exibir resumo das colunas geradas e seus tipos
print('📊 Colunas do dataframe ajustado:')
print(df_ajustado.dtypes)
print(f'\n✅ Shape final: {df_ajustado.shape[0]} linhas x {df_ajustado.shape[1]} colunas')

df_ajustado.head(50)

📊 Colunas do dataframe ajustado:
STATUS_FINAL_NUM                     int64
SISTEMA_CSA                          int64
SISTEMA_DFE                          int64
SISTEMA_DPP                          int64
SISTEMA_SIGAF                        int64
SISTEMA_SIMP                         int64
TIPO_DEMANDA_BUG_IMPEDITIVO          int64
TIPO_DEMANDA_BUG_NAO_IMPEDITIVO      int64
TIPO_DEMANDA_MELHORIA_MEDIA          int64
TIPO_DEMANDA_MELHORIA_PEQUENA        int64
TIPO_DEMANDA_ORIENTACAO              int64
DURACAO_PREVISTA_DIAS                int64
ATRASO_INICIO_DIAS                   int64
DIAS_RESTANTES                       int64
PERC_TEMPO_DECORRIDO               float64
DIAS_SEM_INICIAR                     int64
FLAG_INICIADA                        int64
FLAG_ATRASO_INICIO                   int64
FLAG_NAO_INICIADA_NO_PRAZO           int64
dtype: object

✅ Shape final: 1500 linhas x 19 colunas


,STATUS_FINAL_NUM,SISTEMA_CSA,SISTEMA_DFE,SISTEMA_DPP,SISTEMA_SIGAF,SISTEMA_SIMP,TIPO_DEMANDA_BUG_IMPEDITIVO,TIPO_DEMANDA_BUG_NAO_IMPEDITIVO,TIPO_DEMANDA_MELHORIA_MEDIA,TIPO_DEMANDA_MELHORIA_PEQUENA,TIPO_DEMANDA_ORIENTACAO,DURACAO_PREVISTA_DIAS,ATRASO_INICIO_DIAS,DIAS_RESTANTES,PERC_TEMPO_DECORRIDO,DIAS_SEM_INICIAR,FLAG_INICIADA,FLAG_ATRASO_INICIO,FLAG_NAO_INICIADA_NO_PRAZO
0,0,0,1,0,0,0,1,0,0,0,0,2,0,1,50.0,1,1,0,0
1,0,0,1,0,0,0,1,0,0,0,0,2,2,0,100.0,2,1,1,0
2,0,0,1,0,0,0,1,0,0,0,0,2,0,1,50.0,1,1,0,0
3,0,0,1,0,0,0,1,0,0,0,0,2,0,1,50.0,1,1,0,0
4,0,0,1,0,0,0,1,0,0,0,0,2,0,1,50.0,1,1,0,0
5,0,0,1,0,0,0,1,0,0,0,0,2,0,1,50.0,1,1,0,0
6,0,0,1,0,0,0,1,0,0,0,0,2,0,1,50.0,1,1,0,0
7,0,0,1,0,0,0,1,0,0,0,0,2,0,1,50.0,1,1,0,0
8,0,0,1,0,0,0,1,0,0,0,0,2,0,1,50.0,1,1,0,0
9,0,0,1,0,0,0,1,0,0,0,0,2,0,1,50.0,1,1,0,0


#ANALISE DOS DADOS

In [74]:
# ============================================================
# VERIFICAR O BALANCEAMENTO DA COLUNA TARGET
# ============================================================

balanceamento = df_ajustado['STATUS_FINAL_NUM'].value_counts()
percentual    = df_ajustado['STATUS_FINAL_NUM'].value_counts(normalize=True) * 100

# Exibir resultado
print('Distribuição da variável target:\n')
print(f'🟢 DENTRO DO PRAZO (0): {balanceamento[0]} registros ({percentual[0]:.1f}%)')
print(f'🔴 ATRASO          (1): {balanceamento[1]} registros ({percentual[1]:.1f}%)')

Distribuição da variável target:

🟢 DENTRO DO PRAZO (0): 1110 registros (74.0%)
🔴 ATRASO          (1): 390 registros (26.0%)


#TESTANDO OS MODELOS

In [75]:

# ----------------------------------------------------------
# Separar features (X) da variável target (y)
# X = colunas que o modelo usa para aprender
# y = coluna que o modelo deve prever
# ----------------------------------------------------------


X = df_ajustado.drop(columns=['STATUS_FINAL_NUM'])
y = df_ajustado['STATUS_FINAL_NUM']

# ----------------------------------------------------------
# Dividir em treino (70%) e teste (30%)
# stratify=y garante que a proporção de 0 e 1 seja mantida
# igual nos dois conjuntos — essencial para dados desbalanceados
# random_state=42 garante que a divisão seja reproduzível
# ----------------------------------------------------------
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size    = 0.30,
    stratify     = y,
    random_state = 42
)

# Exibir o resultado da divisão
print(f'Total de registros  : {len(X)}')
print(f'Registros de treino : {len(X_treino)} ({len(X_treino)/len(X)*100:.1f}%)')
print(f'Registros de teste  : {len(X_teste)} ({len(X_teste)/len(X)*100:.1f}%)')
print(f'\nDistribuição do target no treino:\n{y_treino.value_counts()}')
print(f'\nDistribuição do target no teste:\n{y_teste.value_counts()}')

Total de registros  : 1500
Registros de treino : 1050 (70.0%)
Registros de teste  : 450 (30.0%)

Distribuição do target no treino:
STATUS_FINAL_NUM
0    777
1    273
Name: count, dtype: int64

Distribuição do target no teste:
STATUS_FINAL_NUM
0    333
1    117
Name: count, dtype: int64


In [76]:
# ============================================================
# TREINAMENTO DOS MODELOS
# ============================================================

# ============================================================
# TREINAR O MODELO RANDOM FOREST
# ============================================================

# Instanciar o modelo Random Forest
# n_estimators=100 define 100 árvores de decisão
# class_weight='balanced' ajusta automaticamente o peso das classes
# random_state=42 garante reprodutibilidade dos resultados
rf_modelo = RandomForestClassifier(
    n_estimators = 100,
    class_weight = 'balanced',
    random_state = 42
)

# Treinar o modelo com os dados de treino
rf_modelo.fit(X_treino, y_treino)

# Gerar predições com os dados de teste
rf_predicao = rf_modelo.predict(X_teste)


# ============================================================
# AVALIAR E COMPARAR OS MODELOS
# ============================================================


In [77]:
# ============================================================
# TREINAR O NODELO XGBOOST
# ============================================================

# Calcular o peso das classes para lidar com desbalanceamento
# scale_pos_weight = total de negativos / total de positivos
escala_peso = (y_treino == 0).sum() / (y_treino == 1).sum()

# Instanciar o modelo XGBoost
# scale_pos_weight ajusta o peso das classes desbalanceadas
# random_state=42 garante reprodutibilidade dos resultados
xgb_modelo = XGBClassifier(
    scale_pos_weight = escala_peso,
    random_state     = 42,
    eval_metric      = 'logloss'
)

# Treinar o modelo com os dados de treino
xgb_modelo.fit(X_treino, y_treino)

# Gerar predições com os dados de teste
xgb_predicao = xgb_modelo.predict(X_teste)


 Recall e o F1-Score são as métricas mais importantes — é melhor o modelo alertar um possível atraso que não vai acontecer do que deixar passar um atraso real sem aviso.

In [78]:
# ============================================================
# AVALIAR E COMPARAR OS MODELOS
# ============================================================

# Função para calcular e exibir as métricas de cada modelo
def avaliar_modelo(nome, y_teste, y_predicao):
    print(f'\n{"="*50}')
    print(f'  {nome}')
    print(f'{"="*50}')
    print(f'Acurácia  : {accuracy_score(y_teste, y_predicao):.2%}')
    print(f'Precisão  : {precision_score(y_teste, y_predicao):.2%}')
    print(f'Recall    : {recall_score(y_teste, y_predicao):.2%}')
    print(f'F1-Score  : {f1_score(y_teste, y_predicao):.2%}')
    print(f'\nRelatório completo:')
    print(classification_report(y_teste, y_predicao,
                                target_names=['Dentro do Prazo', 'Atraso']))

# Avaliar os dois modelos
avaliar_modelo('RANDOM FOREST', y_teste, rf_predicao)
avaliar_modelo('XGBOOST'      , y_teste, xgb_predicao)


  RANDOM FOREST
Acurácia  : 78.44%
Precisão  : 55.62%
Recall    : 84.62%
F1-Score  : 67.12%

Relatório completo:
                 precision    recall  f1-score   support

Dentro do Prazo       0.93      0.76      0.84       333
         Atraso       0.56      0.85      0.67       117

       accuracy                           0.78       450
      macro avg       0.75      0.80      0.76       450
   weighted avg       0.84      0.78      0.80       450


  XGBOOST
Acurácia  : 77.33%
Precisão  : 53.97%
Recall    : 87.18%
F1-Score  : 66.67%

Relatório completo:
                 precision    recall  f1-score   support

Dentro do Prazo       0.94      0.74      0.83       333
         Atraso       0.54      0.87      0.67       117

       accuracy                           0.77       450
      macro avg       0.74      0.81      0.75       450
   weighted avg       0.84      0.77      0.79       450



In [79]:
# ============================================================
# OTIMIZAÇÃO DE HIPERPARÂMETROS
# Técnica: RandomizedSearchCV
# Mais eficiente que GridSearchCV pois testa combinações
# aleatórias ao invés de todas as combinações possíveis
# ============================================================


# ============================================================
# HIPERPARÂMETROS DO RANDOM FOREST
# ============================================================

# Definir o grid de hiperparâmetros a testar
rf_parametros = {

    # Quantidade de árvores de decisão
    # Mais árvores = mais robusto, porém mais lento
    'n_estimators': [100, 200, 300, 500],

    # Profundidade máxima de cada árvore
    # None = cresce até separar todas as classes (risco de overfitting)
    # Valores menores = modelo mais simples e generalista
    'max_depth': [None, 5, 10, 20, 30],

    # Mínimo de amostras para dividir um nó interno
    # Valores maiores = árvores mais simples, menos overfitting
    'min_samples_split': [2, 5, 10],

    # Mínimo de amostras em um nó folha
    # Valores maiores = modelo mais conservador
    'min_samples_leaf': [1, 2, 4],

    # Quantidade de features consideradas em cada divisão
    # sqrt = raiz quadrada do total de features (padrão)
    # log2 = logaritmo base 2 do total de features
    'max_features': ['sqrt', 'log2'],

    # Peso das classes para lidar com desbalanceamento
    'class_weight': ['balanced', 'balanced_subsample']
}

# Instanciar o modelo base
rf_modelo_otimizado = RandomForestClassifier(random_state=42)

# Instanciar o RandomizedSearchCV
# n_iter=50 testa 50 combinações aleatórias do grid
# cv=5 usa validação cruzada com 5 divisões
# scoring='f1' otimiza pelo F1-Score — ideal para dados desbalanceados
# n_jobs=-1 usa todos os núcleos do processador para acelerar
rf_busca = RandomizedSearchCV(
    estimator          = rf_modelo_otimizado,
    param_distributions = rf_parametros,
    n_iter             = 50,
    cv                 = 5,
    scoring            = 'f1',
    n_jobs             = -1,
    random_state       = 42,
    verbose            = 1
)

# Treinar com os dados de treino
rf_busca.fit(X_treino, y_treino)

# Exibir os melhores hiperparâmetros encontrados
print('Melhores hiperparâmetros — Random Forest:')
print(rf_busca.best_params_)

# ============================================================
# BLOCO 2 — HIPERPARÂMETROS DO XGBOOST
# ============================================================

xgb_parametros = {

    # Quantidade de árvores (rodadas de boosting)
    'n_estimators': [100, 200, 300, 500],

    # Profundidade máxima de cada árvore
    # Valores entre 3 e 10 são os mais comuns
    'max_depth': [3, 5, 7, 10],

    # Taxa de aprendizado — controla o peso de cada árvore
    # Valores menores = aprendizado mais lento e preciso
    'learning_rate': [0.01, 0.05, 0.1, 0.2],

    # Proporção de features usadas por árvore
    # Reduz overfitting ao não usar todas as features sempre
    'colsample_bytree': [0.6, 0.8, 1.0],

    # Proporção de amostras usadas por árvore
    # Reduz overfitting ao não usar todos os registros sempre
    'subsample': [0.6, 0.8, 1.0],

    # Peso mínimo necessário para criar um novo nó folha
    # Valores maiores = modelo mais conservador
    'min_child_weight': [1, 3, 5],

    # Peso das classes desbalanceadas
    # Calculado como: total negativos / total positivos
    'scale_pos_weight': [
        (y_treino == 0).sum() / (y_treino == 1).sum()
    ]
}

# Instanciar o modelo base
xgb_modelo_otimizado = XGBClassifier(
    random_state = 42,
    eval_metric  = 'logloss'
)

# Instanciar o RandomizedSearchCV para o XGBoost
xgb_busca = RandomizedSearchCV(
    estimator           = xgb_modelo_otimizado,
    param_distributions = xgb_parametros,
    n_iter              = 50,
    cv                  = 5,
    scoring             = 'f1',
    n_jobs              = -1,
    random_state        = 42,
    verbose             = 1
)

# Treinar com os dados de treino
xgb_busca.fit(X_treino, y_treino)

# Exibir os melhores hiperparâmetros encontrados
print('\nMelhores hiperparâmetros — XGBoost:')
print(xgb_busca.best_params_)

# ============================================================
# BLOCO 3 — AVALIAR OS MODELOS OTIMIZADOS
# ============================================================

# Gerar predições com os melhores modelos encontrados
rf_melhor_predicao  = rf_busca.best_estimator_.predict(X_teste)
xgb_melhor_predicao = xgb_busca.best_estimator_.predict(X_teste)

# Avaliar os dois modelos otimizados
avaliar_modelo('RANDOM FOREST OTIMIZADO', y_teste, rf_melhor_predicao)
avaliar_modelo('XGBOOST OTIMIZADO'      , y_teste, xgb_melhor_predicao)

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Melhores hiperparâmetros — Random Forest:
{'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 5, 'class_weight': 'balanced'}
Fitting 5 folds for each of 50 candidates, totalling 250 fits

Melhores hiperparâmetros — XGBoost:
{'subsample': 0.8, 'scale_pos_weight': np.float64(2.8461538461538463), 'n_estimators': 200, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.8}

  RANDOM FOREST OTIMIZADO
Acurácia  : 78.22%
Precisão  : 55.14%
Recall    : 87.18%
F1-Score  : 67.55%

Relatório completo:
                 precision    recall  f1-score   support

Dentro do Prazo       0.94      0.75      0.84       333
         Atraso       0.55      0.87      0.68       117

       accuracy                           0.78       450
      macro avg       0.75      0.81      0.76       450
   weighted avg       0.84      0.78      0.79       450


  XGBOOST O

In [84]:
# ============================================================
# PREDIÇÃO DE NOVO REGISTRO
# ============================================================


# ----------------------------------------------------------
# Preencher os dados da nova demanda
# Altere os valores conforme a demanda que deseja testar
# ----------------------------------------------------------
nova_demanda = {
    'ID_DEMANDA'            : 'DEM-999',
    'SISTEMA'               : 'DFE',
    'TIPO_DEMANDA'          : 'MELHORIA_MEDIA',
    'DATA_INICIO_PREVISTA'  : '01/05/2025',
    'DATA_INICIO_REALIZADA' : '11/05/2025',            # None se ainda não iniciada
    'DATA_FIM_PREVISTA'     : '21/05/2025',
    'DATA_REFERENCIA'       : '15/05/2025'     # data do momento da consulta
}

df_nova = pd.DataFrame([nova_demanda])

# ============================================================
# BLOCO 1 — CONVERTER DATAS PARA DATETIME
# ============================================================
colunas_data = [
    'DATA_INICIO_PREVISTA',
    'DATA_INICIO_REALIZADA',
    'DATA_FIM_PREVISTA',
    'DATA_REFERENCIA'
]

for coluna in colunas_data:
    df_nova[coluna] = pd.to_datetime(
        df_nova[coluna], dayfirst=True, errors='coerce'
    )

# ============================================================
# BLOCO 2 — GERAR AS FEATURES
# Mesma lógica aplicada no treinamento — obrigatório
# ============================================================

df_nova['DURACAO_PREVISTA_DIAS'] = (
    df_nova['DATA_FIM_PREVISTA'] - df_nova['DATA_INICIO_PREVISTA']
).dt.days

df_nova['ATRASO_INICIO_DIAS'] = (
    df_nova['DATA_INICIO_REALIZADA'] - df_nova['DATA_INICIO_PREVISTA']
).dt.days.fillna(0)

df_nova['DIAS_RESTANTES'] = (
    df_nova['DATA_FIM_PREVISTA'] - df_nova['DATA_REFERENCIA']
).dt.days

df_nova['PERC_TEMPO_DECORRIDO'] = (
    (df_nova['DATA_REFERENCIA'] - df_nova['DATA_INICIO_PREVISTA']).dt.days /
     df_nova['DURACAO_PREVISTA_DIAS']
) * 100

df_nova['DIAS_SEM_INICIAR'] = (
    df_nova['DATA_REFERENCIA'] - df_nova['DATA_INICIO_PREVISTA']
).dt.days.clip(lower=0)

df_nova['FLAG_INICIADA'] = (
    df_nova['DATA_INICIO_REALIZADA'].notna()
).astype(int)

df_nova['FLAG_ATRASO_INICIO'] = (
    df_nova['ATRASO_INICIO_DIAS'] > 0
).astype(int)

df_nova['FLAG_NAO_INICIADA_NO_PRAZO'] = (
    (df_nova['DATA_REFERENCIA'] >= df_nova['DATA_INICIO_PREVISTA']) &
    (df_nova['DATA_INICIO_REALIZADA'].isna())
).astype(int)

# ============================================================
# BLOCO 3 — APLICAR ONE-HOT ENCODING
# Usar get_dummies e alinhar com as colunas do treino
# para garantir que o modelo receba as mesmas colunas
# ============================================================

# Aplicar One-Hot Encoding nas colunas categóricas
df_nova = pd.get_dummies(df_nova, columns=['SISTEMA', 'TIPO_DEMANDA'], dtype=int)

# Alinhar colunas com o dataset de treino
# Colunas ausentes são preenchidas com 0
# Colunas extras são removidas
df_nova = df_nova.reindex(columns=X_treino.columns, fill_value=0)

# ============================================================
# BLOCO 4 — REMOVER COLUNAS QUE NÃO ENTRAM NO MODELO
# ============================================================
colunas_remover = [
    'ID_DEMANDA',
    'STATUS_FINAL',
    'DATA_INICIO_PREVISTA',
    'DATA_INICIO_REALIZADA',
    'DATA_FIM_PREVISTA',
    'DATA_FIM_REALIZADA',
    'DATA_REFERENCIA'
]

# Remover apenas as colunas que existirem no dataframe
colunas_existentes = [c for c in colunas_remover if c in df_nova.columns]
df_nova = df_nova.drop(columns=colunas_existentes)

# ============================================================
# BLOCO 5 — GERAR PREDIÇÕES E PROBABILIDADES
# ============================================================

# Função para exibir o resultado da predição
def exibir_predicao(nome_modelo, modelo, dados):

    # Predição binária — 0 ou 1
    predicao = modelo.predict(dados)[0]

    # Probabilidade de cada classe em percentual
    probabilidade = modelo.predict_proba(dados)[0]

    resultado = '🔴 RISCO DE ATRASO' if predicao == 1 else '🟢 DENTRO DO PRAZO'

    print(f'\n{"="*50}')
    print(f'  {nome_modelo}')
    print(f'{"="*50}')
    print(f'Resultado         : {resultado}')
    print(f'Prob. No Prazo    : {probabilidade[0]:.2%}')
    print(f'Prob. Atraso      : {probabilidade[1]:.2%}')

# Exibir predição dos dois modelos
exibir_predicao('RANDOM FOREST OTIMIZADO', rf_busca.best_estimator_,  df_nova)
exibir_predicao('XGBOOST OTIMIZADO'      , xgb_busca.best_estimator_, df_nova)


  RANDOM FOREST OTIMIZADO
Resultado         : 🔴 RISCO DE ATRASO
Prob. No Prazo    : 12.94%
Prob. Atraso      : 87.06%

  XGBOOST OTIMIZADO
Resultado         : 🔴 RISCO DE ATRASO
Prob. No Prazo    : 16.88%
Prob. Atraso      : 83.12%
